# 02 - Limpieza y preprocesamiento

**Responsable principal:** Cindy Mishelle Gualim Perez

**Actividad de la guia:** 3 (describir tareas de limpieza/preprocesamiento) y parte de *Descripcion de los datos* (20 pts, valores faltantes/outliers).

**Objetivo de este notebook:** decidir y documentar como se tratan los datos faltantes y ruidosos antes del EDA numerico/visual. Independiente de 01/03/04/05.

In [ ]:
import sys
sys.path.append("..")

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
from src import data_loading as dl

sns.set_theme(style="whitegrid")


## 1. Valores faltantes en landmarks

En este reto es normal que falten landmarks (p. ej. una mano sale del cuadro). Se mide la tasa de NaN por tipo de landmark (`dl.missing_landmark_rate`) para cada secuencia de la muestra fija (`config.SAMPLE_LANDMARK_PATHS`).

In [ ]:
missing_rows = []
for rel_path in config.SAMPLE_LANDMARK_PATHS:
    df = dl.load_landmarks(dl.landmark_path(rel_path))
    for seq_id, seq_df in df.groupby(df.index):
        row = {"sequence_id": seq_id}
        row.update({lt: dl.missing_landmark_rate(seq_df, lt) for lt in config.LANDMARK_TYPES})
        missing_rows.append(row)

missing_df = pd.DataFrame(missing_rows)
missing_df.describe()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=missing_df[config.LANDMARK_TYPES], ax=ax)
ax.set_title("Proporcion de landmarks faltantes por tipo (muestra fija)")
ax.set_ylabel("Proporcion de NaN por secuencia")
fig.savefig(config.ROOT_DIR / "reports" / "figures" / "02_missing_landmarks_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()


## 2. Estrategia de manejo de datos faltantes

Opciones evaluadas:
- **Interpolacion lineal** entre frames validos: funciona bien para huecos cortos, pero en fingerspelling la mano cambia de forma rapido y de manera no lineal entre letras, asi que interpolar huecos largos puede inventar posiciones intermedias que nunca ocurrieron.
- **Descartar frames/secuencias con demasiados NaN**: pierde informacion, pero es la unica opcion razonable cuando el landmark clave (la mano) simplemente no fue detectado durante buena parte de la secuencia -- ahi no hay nada que reconstruir.
- **Forward-fill (ultimo valor valido)**: asume que, entre dos frames consecutivos, es mas probable que la mano se haya quedado quieta o fuera del cuadro que la interpolacion invente. Es el estandar en datos de movimiento continuo capturados frame a frame.</cell id="cell-5">


**Decision:** combinar forward-fill dentro de cada secuencia (`groupby(sequence_id).ffill()`) para huecos cortos, con descarte de secuencias donde una mano este ausente en una proporcion muy alta de frames (umbral a confirmar con los resultados reales de la seccion 1, tentativamente >90%).

**Por que:** en la muestra usada en el notebook 04 (`sample_landmarks.describe()`), las columnas de `face` tienen muy poco faltante (`x_face_0`: 160,278 de 161,461 filas, ~0.7% NaN) mientras que las de `right_hand` tienen mucho mas (`z_right_hand_11`: 73,394 de 161,461, ~54.5% NaN). El problema real de faltantes esta concentrado en las manos -- justo el landmark mas importante para fingerspelling -- y no es ruido disperso que se pueda ignorar. Forward-fill evita inventar movimiento donde no lo hay (a diferencia de interpolar), y descartar secuencias con una mano casi siempre ausente evita meter al analisis secuencias sin señal util. No se interpola porque el movimiento de deletreo cambia de forma rapido entre letras.</cell id="cell-6">


## 3. Secuencias/columnas a descartar

En fingerspelling, los landmarks clave son las **manos** (`left_hand`, `right_hand`). Los landmarks de **cara** (`face`) y **postura** (`pose`) son contexto pero no son la señal principal. Aquí se evalúa:
- Descartar columnas de landmarks menos relevantes: `face` tiene 468 puntos pero no es donde está el deletreo. Evaluamos si redimensionar a solo manos acelera el análisis sin perder información crítica.
- Descartar secuencias corruptas: índices duplicados dentro de un archivo, frames con timestamp faltante, o sequence_id con 0 frames de landmarks válidos después del forward-fill.

In [ ]:
sample_landmarks = dl.load_landmarks(dl.landmark_path(config.SAMPLE_LANDMARK_PATHS[0]))

peso = []
for lt in config.LANDMARK_TYPES:
    n_cols = sum(len(dl.get_landmark_columns(sample_landmarks, lt, coord)) for coord in config.COORDS)
    peso.append({"tipo": lt, "columnas": n_cols})

peso_df = pd.DataFrame(peso)
peso_df["pct_columnas"] = 100 * peso_df["columnas"] / sample_landmarks.shape[1]
peso_df


In [ ]:
frames_por_seq = sample_landmarks.groupby(sample_landmarks.index)["frame"].agg(["size", "nunique"])
print("Secuencias con frames duplicados:", int((frames_por_seq["size"] != frames_por_seq["nunique"]).sum()))

cols_manos_x = dl.get_landmark_columns(sample_landmarks, "left_hand", "x") + dl.get_landmark_columns(sample_landmarks, "right_hand", "x")
frames_con_mano = sample_landmarks[cols_manos_x].notna().any(axis=1).groupby(sample_landmarks.index).sum()
print("Secuencias sin ningun frame con mano detectada:", int((frames_con_mano == 0).sum()))

cols_reducidas = ["frame"]
for lt in ["left_hand", "right_hand", "pose"]:
    for coord in config.COORDS:
        cols_reducidas += dl.get_landmark_columns(sample_landmarks, lt, coord)

print("Columnas originales:", sample_landmarks.shape[1], "-> sin face:", len(cols_reducidas))


**Decision:** para el EDA se trabaja con un subconjunto de columnas: `left_hand`, `right_hand` y `pose` (mas `frame`), descartando las 468 posiciones de `face` en x/y/z.

**Por que:** `face` es ~86% de las columnas del parquet pero no es donde ocurre el deletreo -- la informacion de fingerspelling esta en la forma de la mano, y `pose` se conserva porque ubica los brazos y hombros, que dan el marco de referencia del movimiento. Quitar `face` baja el ancho del DataFrame de 1,630 a poco mas de 200 columnas, lo que hace viable cargar y cruzar varias secuencias en memoria (cada parquet de la muestra son ~1.5 GB).

Sobre secuencias a excluir: se revisan dos casos de corrupcion -- frames repetidos dentro de una misma secuencia (el `frame` deberia ser unico por `sequence_id`) y secuencias donde ninguna mano fue detectada en ningun frame. Estas ultimas no aportan nada al analisis de deletreo y se descartan junto con las que superen el umbral de faltante definido en la seccion 2.

## 4. Guardar datos procesados

Se aplica lo decidido en las secciones 2 y 3 (quedarse con manos + pose, forward-fill por secuencia, descartar secuencias sin mano util) y se guarda el resultado en `data/processed/`, para que los demas notebooks no tengan que releer los ~1.5 GB de cada parquet original.

In [ ]:
UMBRAL_MANO = 0.10  # minimo de frames de la secuencia con al menos una mano detectada


def limpiar_landmarks(df: pd.DataFrame) -> pd.DataFrame:
    """Reduce a manos + pose, descarta secuencias sin mano util y aplica forward-fill por secuencia."""
    cols = ["frame"]
    for lt in ["left_hand", "right_hand", "pose"]:
        for coord in config.COORDS:
            cols += dl.get_landmark_columns(df, lt, coord)

    cols_manos_x = dl.get_landmark_columns(df, "left_hand", "x") + dl.get_landmark_columns(df, "right_hand", "x")
    cobertura_mano = df[cols_manos_x].notna().any(axis=1).groupby(df.index).mean()
    secuencias_validas = cobertura_mano[cobertura_mano >= UMBRAL_MANO].index

    reducido = df.loc[df.index.isin(secuencias_validas), cols]
    return reducido.groupby(reducido.index).ffill()


In [ ]:
config.DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

resumen = []
for rel_path in config.SAMPLE_LANDMARK_PATHS:
    df = dl.load_landmarks(dl.landmark_path(rel_path))
    limpio = limpiar_landmarks(df)
    limpio.to_parquet(config.DATA_PROCESSED_DIR / f"{Path(rel_path).stem}_clean.parquet")

    resumen.append({
        "archivo": Path(rel_path).name,
        "secuencias_orig": df.index.nunique(),
        "secuencias_limpias": limpio.index.nunique(),
        "columnas_orig": df.shape[1],
        "columnas_limpias": limpio.shape[1],
        "pct_nan_antes": 100 * df.isna().mean().mean(),
        "pct_nan_despues": 100 * limpio.isna().mean().mean(),
    })

pd.DataFrame(resumen)


El forward-fill no puede rellenar los NaN que estan **antes** del primer frame con mano detectada (no hay valor previo que arrastrar), asi que `pct_nan_despues` no baja a cero. Esos frames iniciales se dejan como NaN a proposito: rellenarlos hacia atras seria inventar una posicion de mano antes de que la camara la viera.

Los archivos quedan como `data/processed/<file_id>_clean.parquet` (ignorados por git, cada quien los genera corriendo esta seccion).

## 5. Resumen para el informe

TODO: describir en el informe que limpieza se hizo y por que (seccion *Descripcion de los datos*).

TODO